# 03 · Joins: Vendas + Funcionários + Empresas (Caso A)

🎯 **Objetivo:** Responder perguntas de negócio que exigem combinar dados de múltiplas tabelas usando `join`.

**Teoria:** docs/04-dataframes-catalyst-tungsten.md

Duas perguntas que só um join resolve:

1. **Quem mais vendeu?** — `vendas` tem `id_funcionario`, mas o nome está em `funcionarios`
2. **Qual setor vendeu mais em cada período?** — `setor` só existe em `empresas`, então precisamos encadear dois joins

📌 **Conceito:** Join combina linhas de duas tabelas com base em uma coluna em comum (chave). No Spark, o otimizador Catalyst escolhe automaticamente a estratégia mais eficiente (shuffle hash join, broadcast join, etc.).

---
### 🔗 Por que joins são essenciais?

Dados reais raramente estão em uma única tabela. O modelo relacional (e o Spark SQL) organiza dados em tabelas normalizadas ligadas por chaves.

**Nosso modelo:**
```
vendas ───→ funcionarios ───→ empresas
  id_funcionario    id_empresa
```

Para responder "qual setor vendeu mais?", precisamos percorrer essa corrente. Vamos começar pelo join mais simples.


In [ ]:
import sys

sys.path.insert(0, "../scripts")
from lab_utils import get_local_session, layer_path

# Cria a SparkSession e carrega as 3 tabelas da camada Bronze
spark = get_local_session("03-joins")
vendas = spark.read.parquet(layer_path("local", "bronze", "vendas"))
funcionarios = spark.read.parquet(layer_path("local", "bronze", "funcionarios"))
empresas = spark.read.parquet(layer_path("local", "bronze", "empresas"))

✅ **Três DataFrames carregados.** Agora temos:
- `vendas` — transações com `id_funcionario`, `valor`, data
- `funcionarios` — nome, cargo, salário, `id_empresa`
- `empresas` — nome da empresa, `setor`

Vamos conectá-los!


## Ranking de funcionários por vendas

`vendas` só tem `id_funcionario` — um número sem significado para o negócio. Para ver o **nome** do funcionário, precisamos fazer um join com `funcionarios`.

⚠️ **Atenção:** O join padrão do Spark é **inner join** (só linhas que existem nas duas tabelas). Se uma venda referencia um funcionário que não existe na tabela `funcionarios`, essa venda é descartada.

In [ ]:
from pyspark.sql.functions import col
from pyspark.sql.functions import sum as spark_sum

# Join de vendas com funcionarios pela chave id_funcionario
# Depois: agrupa por nome e cargo, soma vendas, ordena do maior para o menor
ranking_funcionarios = (
    vendas.join(funcionarios, "id_funcionario")
    .groupBy("nome_funcionario", "cargo")
    .agg(spark_sum("valor").alias("total_vendas"))
    .orderBy(col("total_vendas").desc())
)
# truncate=False evita que nomes longos sejam cortados
ranking_funcionarios.show(15, truncate=False)

📌 **Interpretação do ranking:**

- Agora temos o **nome** do funcionário ao lado do total de vendas — informação que não existia em nenhuma tabela isoladamente.
- O join combinou `vendas` e `funcionarios` pela chave `id_funcionario`, pareando cada venda com os dados do vendedor.
- O `groupBy` por `nome_funcionario` e `cargo` agregou as vendas de cada pessoa.

💡 **Dica:** O Spark é inteligente: ele empurra o join para antes da agregação, minimizando o volume de dados embaralhados.


## Total de vendas por setor e período (join encadeado)

Agora a pergunta é mais ambiciosa: **qual setor da empresa vendeu mais?** O problema é que `setor` só existe em `empresas`, que se conecta a `vendas` através de `funcionarios`.

A rota é: vendas → funcionarios (via `id_funcionario`) → empresas (via `id_empresa`).

🧠 **Por quê dois joins?** Porque o modelo de dados é normalizado: cada tabela guarda apenas suas próprias colunas, e as chaves estrangeiras fazem a ligação.

In [ ]:
# Encadeia dois joins em sequência: vendas -> funcionarios -> empresas
vendas_por_setor_encadeado = (
    vendas.join(funcionarios, "id_funcionario")
    .join(empresas, "id_empresa")
    .groupBy("setor", "ano", "mes")
    .agg(spark_sum("valor").alias("total_vendas"))
    .orderBy(col("total_vendas").desc())
)
vendas_por_setor_encadeado.show(15)

📌 **Três tabelas, uma resposta:**

- Este resultado só foi possível porque encadeamos dois joins.
- O Spark Catalyst Optimizer funde os dois joins em um único plano de execução, buscando a estratégia mais eficiente.
- Perceba que agora temos `setor`, `ano` e `mes` na mesma linha — dados que vieram de três tabelas diferentes.

🧠 **Para refletir:** O que aconteceria se um funcionário estivesse em `vendas` mas não em `funcionarios`?


## O mesmo resultado, pelo atalho denormalizado

`vendas` já carrega `id_empresa` (o empregador do funcionário daquela venda, gravado no momento da geração dos dados) — então dá pra pular direto para `empresas`, sem passar por `funcionarios`.

Duas rotas, mesmo destino. Compare as somas de `total_vendas` entre as duas células — elas devem ser **idênticas**.

In [ ]:
from pyspark.sql.functions import broadcast

# broadcast() força o Spark a copiar a tabela empresas para TODOS os
# executores, evitando o shuffle caro da tabela vendas (que é grande).
# Só funciona porque empresas tem poucas linhas (~50).
vendas_por_setor_direto = (
    vendas.join(broadcast(empresas), "id_empresa")
    .groupBy("setor", "ano", "mes")
    .agg(spark_sum("valor").alias("total_vendas"))
    .orderBy(col("total_vendas").desc())
)
vendas_por_setor_direto.show(15)

📌 **Comparando as duas abordagens:**

- **Join encadeado** (vendas → funcionarios → empresas): segue o modelo relacional, mas faz 2 shuffles.
- **Join direto** (vendas → empresas com `id_empresa`): apenas 1 join, e com broadcast! Ideal quando a tabela intermediária (`funcionarios`) não é necessária.

⚠️ **Atenção:** O atalho denormalizado só funciona porque o dataset já foi gerado com `id_empresa` em `vendas`. Em sistemas reais, isso nem sempre está disponível.


## Prévia: por que o segundo join foi `broadcast`?

`empresas` tem só ~50 linhas — cabe inteira na memória de cada executor. Com `broadcast()`, o Spark evita o **shuffle** (embaralhamento dos dados pela rede) e simplesmente copia a tabela pequena para todos os nós.

| Estratégia | Sem broadcast | Com broadcast |
|---|---|---|
| Shuffle? | Sim (embaralha `vendas` inteira) | Não (copia `empresas`) |
| Performance | Pode ser lenta com dados grandes | Muito mais rápida |
| Quando usar? | Tabelas grandes dos dois lados | Uma tabela é pequena |

💡 **Dica:** O Spark Catalyst Optimizer já faz broadcast automático para tabelas com menos de 10 MB (configurável via `spark.sql.autoBroadcastJoinThreshold`). O `broadcast()` explícito garante o comportamento independentemente da configuração.

📌 O notebook 04 mostra o **plano de execução** por trás dessa escolha.

In [ ]:
# Encerra a SparkSession
spark.stop()

---
🎉 **Joins concluídos!** Você aprendeu:

- `join` para combinar dados de duas tabelas por uma chave
- **Join encadeado** para navegar por relações de 3 tabelas
- `broadcast()` para otimizar joins com tabelas pequenas
- A diferença entre seguir o modelo relacional vs. usar atalhos denormalizados

📌 **Próximo passo:** Explore os notebooks seguintes para ver como o Spark executa esses planos no cluster!
